# Phase 2: Rolling Averages (Dual Pipeline)

This notebook calculates chronological moving averages for both the **Baseline** dataset and the **Engineered** dataset. By splitting the pipelines, we ensure that if our advanced feature engineering breaks, the baseline model is completely insulated.

In [10]:
import pandas as pd
import numpy as np

### The Master Pipeline Function
This function loads the raw data, calculates the Exponential Moving Averages (EMA), passes through any contextual flags (like Rest Days), merges them against the schedule, and drops the leaky variables.

In [11]:
def process_rolling_pipeline(input_raw_csv, output_csv, is_advanced=False):
    print(f"--- Starting Pipeline for {output_csv} ---")
    
    # 1. Load the specific raw data stream
    df = pd.read_csv(input_raw_csv)
    df = df.sort_values(by=['GAME_DATE', 'GAME_ID']).reset_index(drop=True)
    
    # 2. Define the Spans based on the stream type
    spans = {
        'PTS': 10, 'AST': 10, 'TOV': 10, 'PF': 10,  
        'FG_PCT': 15, 'FG3_PCT': 15, 'FT_PCT': 15,  
        'OREB': 15, 'DREB': 15, 'STL': 15, 'BLK': 15                        
    }
    if is_advanced:
        spans['TS_PCT'] = 15
        spans['eFG_PCT'] = 15
        spans['AST_TOV_RATIO'] = 10
        spans['POSS'] = 10
        spans['OFF_RTG'] = 10
        spans['TOV_PCT'] = 10
        spans['FTR'] = 10
        spans['PCT_PTS_2PT'] = 10
        spans['PCT_PTS_3PT'] = 10
        spans['PCT_PTS_FT'] = 10

    # 3. Calculate Rolling Averages
    def calculate_rolling(group):
        rolled = []
        for col, span in spans.items():
            r = group[[col]].ewm(span=span, min_periods=1).mean().shift(1)
            r.columns = [f"{col}_EMA_{span}"]
            rolled.append(r)
        return pd.concat(rolled, axis=1)

    team_rolling = df.groupby('TEAM_ABBREVIATION', group_keys=False).apply(calculate_rolling)
    
    # 4. Attach Identifiers & Fatigue Context
    keep_cols = ['GAME_ID', 'TEAM_ABBREVIATION']
    if is_advanced:
        keep_cols.extend(['REST_DAYS', 'B2B'])
        
    team_rolling = pd.concat([df[keep_cols], team_rolling], axis=1)
    
    # 5. Merge onto the Schedule
    games_df = pd.read_csv("../data/merged_games_intermediate.csv")
    
    home_rolling = team_rolling.add_prefix('HOME_')
    away_rolling = team_rolling.add_prefix('AWAY_')
    
    games_df = pd.merge(games_df, home_rolling, 
                        left_on=['GAME_ID', 'HOME_TEAM_ABBREVIATION'], 
                        right_on=['HOME_GAME_ID', 'HOME_TEAM_ABBREVIATION'], how='inner')
                        
    games_df = pd.merge(games_df, away_rolling, 
                        left_on=['GAME_ID', 'AWAY_TEAM_ABBREVIATION'], 
                        right_on=['AWAY_GAME_ID', 'AWAY_TEAM_ABBREVIATION'], how='inner')
                        
    games_df = games_df.drop(columns=['HOME_GAME_ID', 'AWAY_GAME_ID'])
    
    # 6. Drop Leaky Columns
    leaky_cols = [
        'HOME_MIN', 'HOME_FGM', 'HOME_FGA', 'HOME_FG_PCT', 'HOME_FG3M', 'HOME_FG3A', 'HOME_FG3_PCT', 
        'HOME_FTM', 'HOME_FTA', 'HOME_FT_PCT', 'HOME_OREB', 'HOME_DREB', 'HOME_REB', 'HOME_AST', 
        'HOME_STL', 'HOME_BLK', 'HOME_TOV', 'HOME_PF', 'HOME_PTS',
        'AWAY_MIN', 'AWAY_FGM', 'AWAY_FGA', 'AWAY_FG_PCT', 'AWAY_FG3M', 'AWAY_FG3A', 'AWAY_FG3_PCT', 
        'AWAY_FTM', 'AWAY_FTA', 'AWAY_FT_PCT', 'AWAY_OREB', 'AWAY_DREB', 'AWAY_REB', 'AWAY_AST', 
        'AWAY_STL', 'AWAY_BLK', 'AWAY_TOV', 'AWAY_PF', 'AWAY_PTS'
    ]
    games_df = games_df.drop(columns=leaky_cols, errors='ignore')
    
    games_df = games_df.dropna().reset_index(drop=True)
    games_df.to_csv(output_csv, index=False)
    print(f"Successfully saved {output_csv} with shape: {games_df.shape}\n")
    return games_df

### Execute Stream A (Baseline Data)

In [12]:
baseline_df = process_rolling_pipeline(
    input_raw_csv="../data/raw_games.csv",
    output_csv="../data/final_training_data_baseline.csv",
    is_advanced=False
)
baseline_df.head()

--- Starting Pipeline for ../data/final_training_data_baseline.csv ---
Successfully saved ../data/final_training_data_baseline.csv with shape: (6124, 31)



,HOME_TEAM_ABBREVIATION,GAME_ID,HOME_GAME_DATE,HOME_MATCHUP,HOME_WL,AWAY_TEAM_ABBREVIATION,AWAY_GAME_DATE,AWAY_MATCHUP,AWAY_WL,HOME_PTS_EMA_10,...,AWAY_AST_EMA_10,AWAY_TOV_EMA_10,AWAY_PF_EMA_10,AWAY_FG_PCT_EMA_15,AWAY_FG3_PCT_EMA_15,AWAY_FT_PCT_EMA_15,AWAY_OREB_EMA_15,AWAY_DREB_EMA_15,AWAY_STL_EMA_15,AWAY_BLK_EMA_15
0,PHI,22100021,2021-10-22,PHI vs. BKN,0,BKN,2021-10-22,BKN @ PHI,1,117.0,...,19.0,13.0,17.0,0.440,0.531,0.565,5.0,39.0,3.0,9.0
1,WAS,22100019,2021-10-22,WAS vs. IND,1,IND,2021-10-22,IND @ WAS,0,98.0,...,29.0,17.0,24.0,0.467,0.362,0.875,8.0,43.0,2.0,10.0
2,SAC,22100026,2021-10-22,SAC vs. UTA,0,UTA,2021-10-22,UTA @ SAC,1,124.0,...,18.0,10.0,19.0,0.440,0.298,0.867,12.0,41.0,6.0,5.0
3,ORL,22100018,2021-10-22,ORL vs. NYK,0,NYK,2021-10-22,NYK @ ORL,1,97.0,...,27.0,19.0,22.0,0.486,0.378,0.704,7.0,48.0,9.0,10.0
4,LAL,22100025,2021-10-22,LAL vs. PHX,0,PHX,2021-10-22,PHX @ LAL,1,114.0,...,23.0,18.0,18.0,0.414,0.378,0.706,11.0,34.0,9.0,3.0


### Execute Stream B (Engineered Data)

In [13]:
advanced_df = process_rolling_pipeline(
    input_raw_csv="../data/engineered_raw_games.csv",
    output_csv="../data/final_training_data_engineered.csv",
    is_advanced=True
)
advanced_df.head()

--- Starting Pipeline for ../data/final_training_data_engineered.csv ---
Successfully saved ../data/final_training_data_engineered.csv with shape: (6124, 55)



,HOME_TEAM_ABBREVIATION,GAME_ID,HOME_GAME_DATE,HOME_MATCHUP,HOME_WL,AWAY_TEAM_ABBREVIATION,AWAY_GAME_DATE,AWAY_MATCHUP,AWAY_WL,HOME_REST_DAYS,...,AWAY_TS_PCT_EMA_15,AWAY_eFG_PCT_EMA_15,AWAY_AST_TOV_RATIO_EMA_10,AWAY_POSS_EMA_10,AWAY_OFF_RTG_EMA_10,AWAY_TOV_PCT_EMA_10,AWAY_FTR_EMA_10,AWAY_PCT_PTS_2PT_EMA_10,AWAY_PCT_PTS_3PT_EMA_10,AWAY_PCT_PTS_FT_EMA_10
0,PHI,22100021,2021-10-22,PHI vs. BKN,0,BKN,2021-10-22,BKN @ PHI,1,2.0,...,0.552486,0.541667,1.46,102.12,101.84,0.1273,0.1548,38.46,49.04,12.50
1,WAS,22100019,2021-10-22,WAS vs. IND,1,IND,2021-10-22,IND @ WAS,0,2.0,...,0.606603,0.561111,1.71,109.56,111.35,0.1552,0.2333,40.98,41.80,17.21
2,SAC,22100026,2021-10-22,SAC vs. UTA,0,UTA,2021-10-22,UTA @ SAC,1,2.0,...,0.548156,0.516484,1.80,95.60,111.92,0.1046,0.1429,48.60,39.25,12.15
3,ORL,22100018,2021-10-22,ORL vs. NYK,0,NYK,2021-10-22,NYK @ ORL,1,2.0,...,0.590349,0.566667,1.42,128.88,107.08,0.1474,0.1810,49.28,36.96,13.77
4,LAL,22100025,2021-10-22,LAL vs. PHX,0,PHX,2021-10-22,PHX @ LAL,1,3.0,...,0.518628,0.494253,1.28,101.48,96.57,0.1774,0.1379,44.90,42.86,12.24


### Extract Schedule Lookup Table
For the future Web Application, we need a flat file that maps Game IDs to Matchups so users can select games from a dropdown.

In [14]:
schedule_df = baseline_df[['GAME_ID', 'HOME_GAME_DATE', 'HOME_TEAM_ABBREVIATION', 'AWAY_TEAM_ABBREVIATION']].copy()
schedule_df = schedule_df.rename(columns={'HOME_GAME_DATE': 'GAME_DATE'})
schedule_df.to_csv("../data/game_schedule.csv", index=False)
schedule_df.head()

,GAME_ID,GAME_DATE,HOME_TEAM_ABBREVIATION,AWAY_TEAM_ABBREVIATION
0,22100021,2021-10-22,PHI,BKN
1,22100019,2021-10-22,WAS,IND
2,22100026,2021-10-22,SAC,UTA
3,22100018,2021-10-22,ORL,NYK
4,22100025,2021-10-22,LAL,PHX
